In [22]:
import pandas as pd
from src.utils import load_json
from classifiers.purpose_me.ollama_inference_updated import load_data, valid_labels

In [23]:
# read in gold labels
examples, invalid = load_data("/Users/alexleto/projects/cross_disciplinary_engagement/data/ann_data/all_citations.csv", valid_labels)
gold_df = pd.DataFrame.from_dict(examples)
gold_df["id"] = gold_df["idx"].astype(str)
gold_df.head()

,idx,csv_row,citation_text,citation_context,label,id
0,0,0,"Klein et al., 2019",The rise of social media and the proliferation...,substantiation,0
1,1,1,"McQuiggan and Lester, 2007",Empathy assesses feelings of sympathy towards ...,related work,1
2,2,2,"Bratton et al., 2019","We introduce SPICED, a new large-scale dataset...",definition,2
3,3,3,"Meng et al., 2018","For WeSTClass and VWS, we used code released b...",use,3
4,4,4,"Nadeem et al., 2019",A text style transfer model is a conditional g...,related work,4


In [24]:
qwen_res = load_json("/Users/alexleto/projects/cross_disciplinary_engagement/results/qwen_inference_new_order.json")["data"]
qwen_res = [{"id":k, "prediction": v["annotation"]["category"]} for k, v in qwen_res.items()]
pred_df = pd.DataFrame.from_dict(qwen_res)
pred_df.head()

,id,prediction
0,6,analysis
1,1,related work
2,7,definition
3,6,analysis
4,2,basis


In [25]:
merged_df = gold_df.merge(pred_df, on="id")
merged_df = merged_df.dropna()
# merged_df.head()
len(merged_df)

949

In [26]:
from sklearn.metrics import classification_report

print(classification_report(merged_df["label"], merged_df["prediction"], labels=valid_labels))

                        precision    recall  f1-score   support

substantiation + basis       0.18      0.09      0.12        43
        substantiation       0.54      0.76      0.63       184
                 basis       0.32      0.55      0.41        44
            definition       0.57      0.44      0.50        36
              analysis       0.55      0.36      0.44       108
                   use       0.79      0.58      0.67       169
          related work       0.69      0.70      0.70       365

              accuracy                           0.61       949
             macro avg       0.52      0.50      0.50       949
          weighted avg       0.62      0.61      0.60       949



In [27]:
import random
merged_df["random"] = merged_df["id"].apply(lambda x: random.choice(valid_labels))
print(classification_report(merged_df["label"], merged_df["random"], labels=valid_labels))


                        precision    recall  f1-score   support

substantiation + basis       0.10      0.28      0.15        43
        substantiation       0.21      0.15      0.18       184
                 basis       0.05      0.16      0.08        44
            definition       0.04      0.17      0.07        36
              analysis       0.10      0.12      0.11       108
                   use       0.14      0.11      0.12       169
          related work       0.35      0.13      0.19       365
substantiation + basis       0.01      0.02      0.01        43
        substantiation       0.15      0.11      0.13       184
                 basis       0.07      0.25      0.11        44
            definition       0.03      0.11      0.05        36
              analysis       0.09      0.12      0.10       108
                   use       0.17      0.14      0.15       169
          related work       0.40      0.15      0.21       365

              accuracy                